In [33]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import cv2
import numpy as np

In [34]:
DATA_DIR = "../data/processed/test"
RAW_DATA_DIR = "../data/raw/COVID-19_Radiography_Dataset/"

print(os.listdir(DATA_DIR))

print(len(os.listdir(os.path.join(DATA_DIR, "Normal"))),
      len(os.listdir(os.path.join(DATA_DIR, "COVID"))),
      len(os.listdir(os.path.join(DATA_DIR, "Lung_Opacity"))),
      len(os.listdir(os.path.join(DATA_DIR, "Viral Pneumonia"))))

CLASSES = [
    "Normal",
    "COVID",
    "Lung_Opacity",
    "Viral Pneumonia"
]

['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
10192 3616 6012 1345


In [35]:
IMG_SIZE = 299

def load_image(image_path):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return image.astype(np.float32)


def load_mask(mask_path):
    mask = cv2.imread(mask_path)
    mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    return (mask > 0).astype(np.float32)

def apply_lung_mask(image, mask):
    return image * mask

# Feature bags

In [36]:
from skimage.feature import local_binary_pattern
import numpy as np

def extract_stats_on_mask(image, mask):
    binary = (mask > 0)
    pixels = image[binary]
    q1, med, q3 = np.percentile(pixels, [25, 50, 75])
    features = {
        "stat_mean":        pixels.mean(),
        "stat_std":         pixels.std(),
        "stat_min":         pixels.min(),
        "stat_max":         pixels.max(),
        "stat_q1":          q1,
        "stat_med":         med,
        "stat_q3":          q3,
        "stat_iqr":         q3 - q1,
        "stat_dark_pixels": float(np.sum(pixels < pixels.mean())),
    }
    return features

def extract_lbp_on_mask(image, mask, P=8, R=1, method="uniform"):
    lbp = local_binary_pattern(image, P, R, method)
    lbp_pixels = lbp[mask > 0]
    n_bins = P + 2
    hist, _ = np.histogram(lbp_pixels, bins=n_bins,
                           range=(0, n_bins), density=True)
    return {f"lbp_r{R}_b{i}": hist[i] for i in range(n_bins)}

def extract_lbp_zones_on_mask(image, mask, P=8, R=1, method="uniform"):
    """LBP séparé gauche / droite pour capturer l'asymétrie spatiale"""
    mid = image.shape[1] // 2
    features = {}
    for side, img_zone, mask_zone in [
        ("left",  image[:, :mid], mask[:, :mid]),
        ("right", image[:, mid:], mask[:, mid:]),
    ]:
        lbp = local_binary_pattern(img_zone, P, R, method)
        lbp_pixels = lbp[mask_zone > 0]
        n_bins = P + 2
        if len(lbp_pixels) == 0:  # masque vide sur cette zone
            hist = np.zeros(n_bins)
        else:
            hist, _ = np.histogram(lbp_pixels, bins=n_bins,
                                   range=(0, n_bins), density=True)
        features.update({f"lbp_{side}_r{R}_b{i}": hist[i] for i in range(n_bins)})
    return features

# Feature extraction pipeline

In [37]:
def extract_all_features(image, mask, feature_extractors):
    features = {}
    for extractor in feature_extractors:
        features.update(extractor(image, mask))
    return features


In [38]:
def create_dataset(feature_extractors, output_path):
    rows = []

    for cls in CLASSES:
        cls_images_path = os.path.join(DATA_DIR, cls)
        cls_masks_path  = os.path.join(RAW_DATA_DIR, cls, "masks")

        cls_images = sorted(os.listdir(cls_images_path))
        cls_masks  = sorted(os.listdir(cls_masks_path))

        for img_name, mask_name in zip(cls_images, cls_masks):
            image = load_image(os.path.join(cls_images_path, img_name))
            mask  = load_mask(os.path.join(cls_masks_path, mask_name))

            row = {"filename": img_name, "class": cls}
            row.update(extract_all_features(image, mask, feature_extractors))
            rows.append(row)

    pd.DataFrame(rows).to_csv(f"{output_path}")

# Train/Test pipeline

In [39]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.utils.class_weight import compute_class_weight


def train_test(df, feature_cols, combo_name=""):
    META_COLS = ["filename", "class"]

    X = df[feature_cols].values
    y = df["class"].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler  = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    class_weight_dict = dict(zip(classes, weights))

    svm = SVC(kernel="linear", C=1.0, gamma="scale",
              class_weight=class_weight_dict, random_state=42)
    svm.fit(X_train, y_train)

    y_pred = svm.predict(X_test)
    """
    print(f"\n{'='*55}")
    print(f"  {combo_name}  ({len(feature_cols)} features)")
    print(f"{'='*55}")
    print(classification_report(y_test, y_pred, target_names=classes))

    cm   = confusion_matrix(y_test, y_pred, labels=classes)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(cmap="Blues")
    plt.title(f"Confusion Matrix — {combo_name}")
    plt.tight_layout()
    plt.show()
    """
    return f1_score(y_test, y_pred, average="macro")

# Ablation study on the feature bags

In [ ]:
from itertools import combinations


META_COLS = ["filename", "class"]
results   = []

ALL_EXTRACTORS = [extract_stats_on_mask, extract_lbp_on_mask, extract_lbp_zones_on_mask]

all_combos = [
    combo
    for r in range(1, len(ALL_EXTRACTORS) + 1)
    for combo in combinations(ALL_EXTRACTORS, r)
]

for combo in all_combos:
    print("Exporting features....")
    combo_name = "+".join(fn.__name__ for fn in combo)
    df = create_dataset(feature_extractors=combo, output_path=os.path.join("..", "data", "features", f"{combo_name}.csv"))

In [41]:
results = []

for combo in all_combos:
    combo_name = "+".join(fn.__name__ for fn in combo)
    df = pd.read_csv(f"../data/features/{combo_name}.csv", index_col=0)
    feature_cols = [c for c in df.columns if c not in META_COLS and not c.startswith("Unnamed")]
    macro_f1     = train_test(df, feature_cols, combo_name=combo_name)

    results.append({
        "combination" : combo_name,
        "n_features"  : len(feature_cols),
        "macro_f1"    : round(macro_f1, 4),
    })

# ── Résumé final ──────────────────────────────────────────────────────────────

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
print("\n" + "="*55)
print("  CLASSEMENT FINAL")
print("="*55)
print(results_df.to_string(index=False))


  CLASSEMENT FINAL
                                                        combination  n_features  macro_f1
extract_stats_on_mask+extract_lbp_on_mask+extract_lbp_zones_on_mask          39    0.6151
                    extract_stats_on_mask+extract_lbp_zones_on_mask          29    0.6017
                          extract_stats_on_mask+extract_lbp_on_mask          19    0.5828
                      extract_lbp_on_mask+extract_lbp_zones_on_mask          30    0.5786
                                          extract_lbp_zones_on_mask          20    0.5579
                                                extract_lbp_on_mask          10    0.5338
                                              extract_stats_on_mask           9    0.4712
